In [1]:
# ==============================================================================
# COMPATIFI V5A — ASSISTANT OBJECTIVE MODEL TEST
# ==============================================================================
#
# V5A:
#   Conversation
#       ↓
#   Assistant Objective
#
# Input:
#   Domain
#   Relationship
#   Conversation
#
# Output:
#   primary_objective
#   secondary_objective
#   priority
#   reason
#
# Model:
#   V5A_Final_Merged_Model
#
# ==============================================================================


# ==============================================================================
# SECTION 1: IMPORTS
# ==============================================================================

import json
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)


# ==============================================================================
# SECTION 2: MODEL PATH
# ==============================================================================

MODEL_PATH = "./V5A_Final_Merged_Model"

MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 180


# ==============================================================================
# SECTION 3: HARDWARE
# ==============================================================================

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required.")

GPU_NAME = torch.cuda.get_device_name(0)

print("=" * 80)
print("COMPATIFI V5A TEST")
print("=" * 80)
print(f"GPU   : {GPU_NAME}")
print(f"MODEL : {MODEL_PATH}")
print("=" * 80)


# ==============================================================================
# SECTION 4: COMPUTE DTYPE
# ==============================================================================

if torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE = torch.bfloat16
else:
    COMPUTE_DTYPE = torch.float16


print(f"Compute dtype: {COMPUTE_DTYPE}")


# ==============================================================================
# SECTION 5: LOAD TOKENIZER
# ==============================================================================

print()
print("=" * 80)
print("LOADING TOKENIZER")
print("=" * 80)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"


# ==============================================================================
# SECTION 6: LOAD MERGED V5A MODEL
# ==============================================================================

print()
print("=" * 80)
print("LOADING V5A MERGED MODEL")
print("=" * 80)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    trust_remote_code=True,
)

model.eval()

print("V5A merged model loaded successfully.")




/home/ubuntu/V5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


COMPATIFI V5A TEST
GPU   : NVIDIA L40S
MODEL : ./V5A_Final_Merged_Model
Compute dtype: torch.bfloat16

LOADING TOKENIZER

LOADING V5A MERGED MODEL


Loading checkpoint shards: 100%|██████████| 4/4 [00:11<00:00,  2.89s/it]

V5A merged model loaded successfully.


In [2]:


# ==============================================================================
# SECTION 7: V5A SYSTEM PROMPT
# ==============================================================================

SYSTEM_PROMPT = """
You are Compatifi V5A.

Your task is to predict what the assistant should try to accomplish
in the current conversation.

Predict the assistant's objective, NOT the user's long-term goal.

Identify:

- primary_objective
- secondary_objective
- priority
- reason

The primary objective is the most important thing the assistant
should accomplish next.

The secondary objective is a supporting objective.

The priority indicates how important the objective is.

The reason must be short and based only on information present
in the conversation.

Always provide one primary_objective.

If there is no meaningful secondary objective, use:
"None"

Return ONLY valid JSON.

Required format:

{
  "primary_objective": "...",
  "secondary_objective": "...",
  "priority": "...",
  "reason": "..."
}

Do NOT:

- generate the final reply
- give advice directly
- generate conversation messages
- extract memories
- predict personality
- predict long-term goals
- invent information
"""

In [3]:


# ==============================================================================
# SECTION 8: TEST CONVERSATIONS
# ==============================================================================

test_cases = [

    # --------------------------------------------------------------------------
    # TEST 1 — REDUCE ANXIETY
    # --------------------------------------------------------------------------

    {
        "name": "Interview Anxiety",

        "domain": "relationship",

        "relationship": "Friend",

        "conversation": """
Friend: You seem worried lately.
User: I have my final interview tomorrow and I'm really nervous.
Friend: You've prepared for weeks.
User: I know, but I keep thinking I'll mess it up.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 2 — DECISION SUPPORT
    # --------------------------------------------------------------------------

    {
        "name": "Family Decision",

        "domain": "relationship",

        "relationship": "Partner",

        "conversation": """
Partner: We need to decide where to spend Christmas this year.
User: I don't know whether we should visit your parents or mine.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 3 — PRACTICAL GUIDANCE
    # --------------------------------------------------------------------------

    {
        "name": "Resume Help",

        "domain": "career",

        "relationship": "Career Coach",

        "conversation": """
Career Coach: How is your resume coming along?
User: I'm struggling to describe my leadership experience.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 4 — EMOTIONAL SUPPORT
    # --------------------------------------------------------------------------

    {
        "name": "Bad Day",

        "domain": "relationship",

        "relationship": "Friend",

        "conversation": """
Friend: How was your day?
User: Honestly, it was terrible.
Friend: What happened?
User: Everything went wrong at work and I'm completely exhausted.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 5 — PROBLEM SOLVING
    # --------------------------------------------------------------------------

    {
        "name": "Work Problem",

        "domain": "work",

        "relationship": "Colleague",

        "conversation": """
Colleague: The project deadline was moved to Friday.
User: That's a problem. We still have three major tasks unfinished.
Colleague: We need to figure out how to finish everything.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 6 — PLANNING
    # --------------------------------------------------------------------------

    {
        "name": "Weekend Planning",

        "domain": "relationship",

        "relationship": "Friend",

        "conversation": """
Friend: We haven't done anything together for weeks.
User: Yeah, we've both been busy.
Friend: We should do something this weekend.
User: Definitely. Let's plan something.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 7 — MAINTAIN RAPPORT
    # --------------------------------------------------------------------------

    {
        "name": "Casual Conversation",

        "domain": "relationship",

        "relationship": "Friend",

        "conversation": """
Friend: What are you doing?
User: Just watching TV.
Friend: Anything good?
User: Just a comedy.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 8 — ENCOURAGEMENT
    # --------------------------------------------------------------------------

    {
        "name": "Learning Progress",

        "domain": "personal",

        "relationship": "Mentor",

        "conversation": """
Mentor: How is your programming practice going?
User: It's difficult, but I've been practicing every day.
Mentor: That's good progress.
User: Sometimes I feel like I'm not improving fast enough.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 9 — CONFLICT
    # --------------------------------------------------------------------------

    {
        "name": "Friendship Conflict",

        "domain": "relationship",

        "relationship": "Friend",

        "conversation": """
Friend: You didn't reply to me yesterday.
User: I was busy.
Friend: You always say that.
User: I'm sorry. I didn't mean to ignore you.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 10 — INFORMATION SHARING
    # --------------------------------------------------------------------------

    {
        "name": "Simple Question",

        "domain": "general",

        "relationship": "Friend",

        "conversation": """
Friend: Do you know what time the movie starts?
User: I think it's at eight.
Friend: Are you sure?
User: Let me check.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 11 — MOTIVATION
    # --------------------------------------------------------------------------

    {
        "name": "Low Motivation",

        "domain": "personal",

        "relationship": "Friend",

        "conversation": """
Friend: Have you started your project yet?
User: No, I keep putting it off.
Friend: What's stopping you?
User: I just don't feel motivated.
"""
    },


    # --------------------------------------------------------------------------
    # TEST 12 — SUPPORT + NEXT STEPS
    # --------------------------------------------------------------------------

    {
        "name": "Relationship Problem",

        "domain": "relationship",

        "relationship": "Partner",

        "conversation": """
Partner: I feel like we've been arguing a lot lately.
User: I don't want things to keep going this way.
Partner: Me neither.
User: I think we need to figure out what's causing these arguments.
"""
    },

]



In [4]:

# ==============================================================================
# SECTION 9: GENERATION FUNCTION
# ==============================================================================

def generate_v5a(test_case):

    user_content = f"""
Domain: {test_case["domain"]}

Relationship: {test_case["relationship"]}

Instruction: Predict the assistant objective

Conversation:
{test_case["conversation"].strip()}
"""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            pad_token_id=tokenizer.eos_token_id,

            eos_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    return response


# ==============================================================================
# SECTION 10: JSON VALIDATION
# ==============================================================================

def validate_v5a_output(response):

    required_fields = [
        "primary_objective",
        "secondary_objective",
        "priority",
        "reason",
    ]

    try:

        data = json.loads(response)

    except json.JSONDecodeError:

        return False, "Invalid JSON", None

    if not isinstance(data, dict):

        return False, "Output is not a JSON object", None

    missing = [
        field
        for field in required_fields
        if field not in data
    ]

    if missing:

        return False, f"Missing fields: {missing}", data

    if not data["primary_objective"]:

        return False, "primary_objective is empty", data

    if not data["priority"]:

        return False, "priority is empty", data

    if not data["reason"]:

        return False, "reason is empty", data

    return True, "Valid V5A output", data


# ==============================================================================
# SECTION 11: RUN TESTS
# ==============================================================================

print()
print("=" * 80)
print("STARTING V5A TESTS")
print("=" * 80)


valid_count = 0


for index, test_case in enumerate(test_cases, 1):

    print()
    print("=" * 80)
    print(f"TEST {index}: {test_case['name']}")
    print("=" * 80)

    print()
    print("DOMAIN:")
    print(test_case["domain"])

    print()
    print("RELATIONSHIP:")
    print(test_case["relationship"])

    print()
    print("CONVERSATION:")
    print(test_case["conversation"].strip())

    print()
    print("-" * 80)
    print("V5A OUTPUT")
    print("-" * 80)

    response = generate_v5a(test_case)

    print(response)

    print()
    print("-" * 80)
    print("VALIDATION")
    print("-" * 80)

    is_valid, message, parsed = validate_v5a_output(response)

    if is_valid:

        valid_count += 1

        print("✅", message)

        print()
        print("Primary Objective   :", parsed["primary_objective"])
        print("Secondary Objective :", parsed["secondary_objective"])
        print("Priority            :", parsed["priority"])
        print("Reason              :", parsed["reason"])

    else:

        print("❌", message)


# ==============================================================================
# SECTION 12: FINAL SUMMARY
# ==============================================================================

print()
print("=" * 80)
print("V5A TEST SUMMARY")
print("=" * 80)

print(f"Total tests : {len(test_cases)}")
print(f"Valid       : {valid_count}")
print(f"Invalid     : {len(test_cases) - valid_count}")

accuracy = valid_count / len(test_cases) * 100

print(f"JSON format success : {accuracy:.1f}%")

print("=" * 80)
print("V5A TEST COMPLETE")
print("=" * 80)


STARTING V5A TESTS

TEST 1: Interview Anxiety

DOMAIN:
relationship

RELATIONSHIP:
Friend

CONVERSATION:
Friend: You seem worried lately.
User: I have my final interview tomorrow and I'm really nervous.
Friend: You've prepared for weeks.
User: I know, but I keep thinking I'll mess it up.

--------------------------------------------------------------------------------
V5A OUTPUT
--------------------------------------------------------------------------------


/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/ubuntu/V5/.venv/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


<think>

</think>

{"primary_objective":"Reduce Anxiety","secondary_objective":"None","priority":"High","reason":"The user is expressing high anxiety about an upcoming interview tomorrow.","memories":null}

--------------------------------------------------------------------------------
VALIDATION
--------------------------------------------------------------------------------
❌ Invalid JSON

TEST 2: Family Decision

DOMAIN:
relationship

RELATIONSHIP:
Partner

CONVERSATION:
Partner: We need to decide where to spend Christmas this year.
User: I don't know whether we should visit your parents or mine.

--------------------------------------------------------------------------------
V5A OUTPUT
--------------------------------------------------------------------------------
<think>

</think>

{"primary_objective":"Support Decision Making","secondary_objective":"None","priority":"Medium","reason":"The user is torn between two holiday travel options and needs help making a choice.","memorie